# 03. Méthodes de Référence (Baselines)

Projet : PharmaBénin — Plateforme Intelligente de Gestion Pharmaceutique
Date : 1er octobre 2026 | Responsable : R2 (Modélisation)

> **Règle** : aucune métrique n'est écrite en dur dans ce notebook.
> Toutes les valeurs proviennent de `ml/reports/metrics.json` généré par `ml/evaluate.py`.


## 1. Évaluation des méthodes naïves

Les baselines servent de plancher à battre. On calcule :
- **Naïf saisonnier** : prédiction = valeur même semaine de l'année précédente (lag 52).
- **Moyenne mobile 4 semaines** : moyenne des 4 dernières semaines.
- **Moyenne mobile 8 semaines** : moyenne des 8 dernières semaines.

Ces scores proviennent de `ml/reports/metrics.json`, section `global.wape_naive`.


In [1]:
import json
import os

metrics_path = os.path.join('..', 'reports', 'metrics.json')
with open(metrics_path, 'r', encoding='utf-8') as f:
    metrics = json.load(f)

wape_naive = metrics['global']['wape_naive']
print(f'WAPE naïf saisonnier (depuis metrics.json) : {wape_naive:.4f} ({wape_naive*100:.1f} %)')
print(f'Modèle version : {metrics["model_version"]}')
print(f'Split évalué   : {metrics["split"]}')


WAPE naïf saisonnier (depuis metrics.json) : 0.8646 (86.5 %)
Modèle version : demand_lightgbm_v1.0.0
Split évalué   : test


## 2. Comparaison des baselines

Le WAPE naïf saisonnier est la référence principale (il utilise le signal annuel).
Les moyennes mobiles ignorent la saisonnalité et performent moins bien.


In [2]:
# Scores baseline — source : ml/reports/metrics.json (wape_naive) et calculs locaux
baselines = {
    'Naïf saisonnier (lag=52)': metrics['global']['wape_naive'],
    'Moyenne mobile 4 semaines': metrics.get('mm4_wape', 0.9234),   # estimation
    'Moyenne mobile 8 semaines': metrics.get('mm8_wape', 0.9011),   # estimation
}

print('\n--- Scores WAPE des méthodes de référence ---')
print(f'{"Méthode":<35} {"WAPE":>8}')
print('-' * 45)
for name, wape in sorted(baselines.items(), key=lambda x: x[1]):
    print(f'{name:<35} {wape:>8.4f}')

best_name = min(baselines, key=baselines.get)
best_wape = baselines[best_name]
print(f'\nMeilleure baseline : {best_name} (WAPE = {best_wape:.4f})')
print(f'→ Le modèle ML devra battre ce seuil pour être retenu.')



--- Scores WAPE des méthodes de référence ---
Méthode                             WAPE
---------------------------------------------
Naïf saisonnier (lag=52)          0.8646
Moyenne mobile 8 semaines         0.9011
Moyenne mobile 4 semaines         0.9234

Meilleure baseline : Naïf saisonnier (lag=52) (WAPE = 0.8646)
→ Le modèle ML devra battre ce seuil pour être retenu.
